In [1]:
!git clone https://github.com/rajatdas-cs/LPG-SAM

Cloning into 'LPG-SAM'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 149 (delta 28), reused 133 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 30.45 MiB | 40.18 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [4]:
import os
import shutil

# Set working directory
os.chdir('/kaggle/working')

# Create checkpoints directory
os.makedirs('LPG-SAM/checkpoints', exist_ok=True)

# Source: dataset folder inside Kaggle input
src_data_path = '/kaggle/input/datasets/hirotakanifuji/frangi-cache/data'

# Destination inside repo
dst_data_path = '/kaggle/working/LPG-SAM/data'

# Copy entire folder
if not os.path.exists(dst_data_path):
    shutil.copytree(src_data_path, dst_data_path)
    print("Data folder copied successfully.")
else:
    print("Data folder already exists, skipping copy.")

# Optional: verify
print("\nContents of LPG-SAM:")
print(os.listdir('/kaggle/working/LPG-SAM'))

Data folder copied successfully.

Contents of LPG-SAM:
['data', 'configs', 'checkpoints', 'project_details.md', 'fov_mask.png', 'FINDINGS.md', '.git', 'src', 'frangi_check', 'sanity_checked.png', 'sanity_check.ipynb', 'requirements.txt', '.gitignore', 'report', 'scripts', 'run.md']


In [5]:
import os
import shutil

# Source: Kaggle dataset path
src_path = '/kaggle/input/datasets/hirotakanifuji/medsam/medsam_vit_b.pth'  # adjust dataset name if needed

# Destination: your repo checkpoints folder
dst_dir = '/kaggle/working/LPG-SAM/checkpoints/'
dst_path = os.path.join(dst_dir, 'medsam_vit_b.pth')

# Ensure destination directory exists
os.makedirs(dst_dir, exist_ok=True)

# Copy file
shutil.copy(src_path, dst_path)

print(f"Checkpoint copied to: {dst_path}")

Checkpoint copied to: /kaggle/working/LPG-SAM/checkpoints/medsam_vit_b.pth


In [8]:
import os
import shutil

# Source: Kaggle dataset path
src_path = '/kaggle/input/datasets/hirotakanifuji/our-checkpoint/frangi-updated-cldice-wt.pt'  # adjust dataset name if needed

# Destination: your repo checkpoints folder
dst_dir = '/kaggle/working/LPG-SAM/checkpoints/'
dst_path = os.path.join(dst_dir, 'frangi-updated-cldice-wt.pt')

# Ensure destination directory exists
os.makedirs(dst_dir, exist_ok=True)

# Copy file
shutil.copy(src_path, dst_path)

print(f"Checkpoint copied to: {dst_path}")

Checkpoint copied to: /kaggle/working/LPG-SAM/checkpoints/frangi-updated-cldice-wt.pt


In [9]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

  Preparing metadata (setup.py) ... done


In [10]:
%cd /kaggle/working/LPG-SAM
!python -m src.adapter
!python -m src.losses
!python -m src.metrics
!python -m scripts.smoke_test

/kaggle/working/LPG-SAM
Device: CUDA: Tesla T4 (15.6 GB)

[1/6] PriorProjector shape check ...
      ok: (2, 1, 1024, 1024) -> (2, 256, 64, 64)
[2/6] FiLMHead shape check ...
      ok: gamma (2, 256, 64, 64), beta (2, 256, 64, 64)
[3/6] TokenHead shape check ...
      ok: tokens (2, 4, 256)
[4/6] LatentPriorAdapter end-to-end shape check ...
[LatentPriorAdapter] trainable: 656.48K / total: 656.48K (100.00%)
      ok: z_mod (2, 256, 64, 64), sparse (2, 4, 256)
[5/6] alpha=0 invariant (z_modulated == z_image) ...
      max |z_mod - z_image| = 0.00e+00
      ok: adapter is a perfect no-op at alpha=0
[6/6] Gradient flow check ...
      ok: alpha.grad = -1.3956e+02

[OK] All adapter checks passed.
Device: CUDA: Tesla T4 (15.6 GB)

[1/4] dice_loss ...
      ok: dice = 0.9587
[2/4] bce_loss ...
      ok: bce  = 0.8035
[3/4] soft_cldice_loss ...
      ok: cldice = 0.9583
[4/4] CompositeLoss + gradient flow ...
      total = 1.1686
      components: dice=0.9587 bce=0.8035 cldice=0.9583
      ok

**OUR MODEL ON FIVES**

In [19]:
import os, sys

REPO = "/kaggle/working/LPG-SAM"
os.chdir(REPO)                        # all relative paths resolve from here
sys.path.insert(0, REPO)

import json, csv, zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")                 # no display on Kaggle
import matplotlib.pyplot as plt
from PIL import Image
from skimage.morphology import binary_closing, disk
from tqdm.notebook import tqdm

# project modules
from src.adapter    import LatentPriorAdapter
from src.sam_wrapper import FrozenSAM, SAM_INPUT_SIZE
from src.metrics    import MetricAccumulator, logits_to_binary
import yaml

# ── paths ──────────────────────────────────────────────────────────────────────
CFG_PATH   = Path("configs/config.yaml")
CKPT_PATH  = Path("checkpoints/frangi-updated-cldice-wt.pt")
DATA_ROOT  = Path("data/processed/FIVES")
SPLIT_FILE = DATA_ROOT / "test.txt"

OUT_DIR    = Path("/kaggle/working/fives_eval")
OVERLAY_DIR = OUT_DIR / "overlays"
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

# ── config ─────────────────────────────────────────────────────────────────────
with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

THRESHOLD      = cfg["eval"]["binarize_threshold"]   # 0.5
CLOSING_RADIUS = cfg["eval"].get("closing_radius", 2)
NUM_TOKENS     = cfg["model"]["num_tokens"]           # 4

# ── device ─────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"Test split: {SPLIT_FILE}")

stems = [l.strip() for l in SPLIT_FILE.read_text().splitlines() if l.strip()]
print(f"Test images: {len(stems)}")


Device: cuda
Checkpoint: checkpoints/frangi-updated-cldice-wt.pt
Test split: data/processed/FIVES/test.txt
Test images: 200


In [20]:
print("Loading frozen MedSAM ...")
sam = FrozenSAM(checkpoint_path=cfg["paths"]["sam_checkpoint"]).to(device)
sam.eval()

print("Loading LatentPriorAdapter ...")
adapter = LatentPriorAdapter(num_tokens=NUM_TOKENS).to(device)
ckpt    = torch.load(CKPT_PATH, map_location=device)
adapter.load_state_dict(ckpt["adapter_state_dict"])
adapter.eval()

print(f"Checkpoint epoch : {ckpt.get('epoch', '?')}")
print(f"Checkpoint val   : {ckpt.get('val_metrics', {})}")
print(f"Adapter alpha    : {adapter.alpha.item():+.4f}")

Loading frozen MedSAM ...
Loading LatentPriorAdapter ...
Checkpoint epoch : 26
Checkpoint val   : {'dice': 0.84037040678926, 'iou': 0.7309048963019397, 'cldice': 0.8705494941179432, 'betti0_error': 91.63333333333334, 'n': 60}
Adapter alpha    : +0.1784


In [13]:
# helper: 5-panel figure
def save_overlay(path, img_np, frangi_np, gt_np, pred_np, dice, cldice):
    """
    img_np   : (H, W, 3) float32 [0,1]
    frangi_np: (H, W)    float32 [0,1]
    gt_np    : (H, W)    bool
    pred_np  : (H, W)    bool
    """
    H, W = gt_np.shape
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    fig.suptitle(f"{path.stem}    Dice={dice:.3f}  clDice={cldice:.3f}",
                 fontsize=11, y=1.01)

    axes[0].imshow(img_np);                     axes[0].set_title("Image")
    axes[1].imshow(frangi_np, cmap="hot",
                   vmin=0, vmax=1);             axes[1].set_title("Frangi")
    axes[2].imshow(gt_np,     cmap="gray",
                   vmin=0, vmax=1);             axes[2].set_title("Ground Truth")
    axes[3].imshow(pred_np,   cmap="gray",
                   vmin=0, vmax=1);             axes[3].set_title("Prediction")

    overlay = np.zeros((H, W, 3), dtype=np.float32)
    overlay[..., 1] = np.logical_and(pred_np,  gt_np ).astype(np.float32)  # TP green
    overlay[..., 0] = np.logical_and(pred_np, ~gt_np ).astype(np.float32)  # FP red
    overlay[..., 2] = np.logical_and(~pred_np, gt_np ).astype(np.float32)  # FN blue
    axes[4].imshow(overlay);                    axes[4].set_title("TP=G  FP=R  FN=B")

    for ax in axes:
        ax.axis("off")

    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.close(fig)


# category map  (last char of stem suffix: A=AMD, D=DR, G=Glaucoma, N=Normal)
CAT_MAP = {"A": "AMD", "D": "Diabetic_Retinopathy", "G": "Glaucoma", "N": "Normal"}

acc = MetricAccumulator()
rows = []   # per-image CSV rows

with torch.no_grad():
    for stem in tqdm(stems, desc="Evaluating", ncols=90):
        # ── load image ──────────────────────────────────────────────────────
        img_path  = DATA_ROOT / "images_1024"  / f"{stem}.png"
        mask_path = DATA_ROOT / "masks_1024"   / f"{stem}.png"
        fng_path  = DATA_ROOT / "frangi_cache" / f"{stem}.npy"

        img_pil  = Image.open(img_path ).convert("RGB")
        mask_pil = Image.open(mask_path).convert("L")
        frangi   = np.load(fng_path).astype(np.float32)   # (H, W)

        img_np   = np.array(img_pil,  dtype=np.float32) / 255.0   # (H, W, 3)
        mask_np  = np.array(mask_pil, dtype=np.float32) / 255.0   # (H, W)

        # ── build tensors ───────────────────────────────────────────────────
        # image : (1, 3, 1024, 1024) in [0,1]   — SAM expects this range
        img_t    = torch.from_numpy(img_np.transpose(2, 0, 1)
                                    ).unsqueeze(0).to(device)
        frangi_t = torch.from_numpy(frangi
                                    ).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,H,W)

        # ── forward pass ────────────────────────────────────────────────────
        z_image       = sam.encode_image(img_t)
        z_mod, tokens = adapter(z_image, frangi_t)
        dense_default = sam.zero_dense_prompt(1, device)

        masks_low, _  = sam.decode_masks(
            image_embeddings       = z_mod,
            sparse_prompt_embeddings = tokens,
            dense_prompt_embeddings  = dense_default,
            multimask_output         = False,
        )
        # upsample 256→1024
        mask_logits = F.interpolate(masks_low, size=(1024, 1024),
                                    mode="bilinear", align_corners=False)

        # ── binarize ────────────────────────────────────────────────────────
        logits_np = mask_logits[0].cpu().numpy()              # (1, H, W)
        pred_bin  = logits_to_binary(logits_np, threshold=THRESHOLD)  # (H, W) bool
        if CLOSING_RADIUS > 0:
            pred_bin = binary_closing(pred_bin, footprint=disk(CLOSING_RADIUS))

        gt_bin = mask_np > 0.5                                # (H, W) bool

        # ── metrics ─────────────────────────────────────────────────────────
        acc.update(pred_bin, gt_bin, name=stem)
        last = acc.per_image_table()[-1]

        category = CAT_MAP.get(stem[-1], "Unknown")
        row = {**last, "category": category}
        rows.append(row)

        # ── save overlay ─────────────────────────────────────────────────────
        save_overlay(
            path      = OVERLAY_DIR / f"{stem}.png",
            img_np    = img_np,
            frangi_np = frangi,
            gt_np     = gt_bin,
            pred_np   = pred_bin,
            dice      = last["dice"],
            cldice    = last["cldice"],
        )

print(f"\nOverlays saved to {OVERLAY_DIR}  ({len(list(OVERLAY_DIR.glob('*.png')))} files)")


Evaluating:   0%|                                                 | 0/200 [00:00<?, ?it/s]


Overlays saved to /kaggle/working/fives_eval/overlays  (200 files)


In [14]:
summary = acc.summary()

# ── aggregate table ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  FIVES Test Set  —  200 images  —  epoch-26 checkpoint")
print("=" * 60)
print(f"  Dice        : {summary['dice']:.4f}")
print(f"  IoU         : {summary['iou']:.4f}")
print(f"  clDice      : {summary['cldice']:.4f}   ← headline")
print(f"  Betti-0 err : {summary['betti0_error']:.2f}")
print(f"  n           : {summary['n']}")

# ── per-category breakdown ───────────────────────────────────────────────────
print("\n  Per-category breakdown:")
print(f"  {'Category':<22} {'n':>4}  {'Dice':>6}  {'IoU':>6}  {'clDice':>7}  {'B0 err':>7}")
print("  " + "-" * 62)
for cat_key, cat_name in [("AMD","AMD"),("Diabetic_Retinopathy","DR"),
                           ("Glaucoma","Glaucoma"),("Normal","Normal")]:
    cat_rows = [r for r in rows if r["category"] == cat_key]
    if not cat_rows:
        continue
    n   = len(cat_rows)
    d   = np.mean([r["dice"]         for r in cat_rows])
    iou = np.mean([r["iou"]          for r in cat_rows])
    cl  = np.mean([r["cldice"]       for r in cat_rows])
    b0  = np.mean([r["betti0_error"] for r in cat_rows])
    print(f"  {cat_name:<22} {n:>4}  {d:>6.4f}  {iou:>6.4f}  {cl:>7.4f}  {b0:>7.2f}")

# ── worst / best ─────────────────────────────────────────────────────────────
sorted_rows = sorted(rows, key=lambda r: r["cldice"])
print(f"\n  Worst case : {sorted_rows[0]['name']:<20}  clDice={sorted_rows[0]['cldice']:.4f}")
print(f"  Best  case : {sorted_rows[-1]['name']:<20}  clDice={sorted_rows[-1]['cldice']:.4f}")

# ── save per-image CSV ───────────────────────────────────────────────────────
csv_path = OUT_DIR / "per_image.csv"
fieldnames = ["name", "category", "dice", "iou", "cldice", "betti0_error"]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(sorted_rows)   # sorted worst-first by clDice
print(f"\n  CSV  → {csv_path}")

# ── save summary JSON ────────────────────────────────────────────────────────
json_path = OUT_DIR / "summary.json"
with open(json_path, "w") as f:
    json.dump({
        "checkpoint": str(CKPT_PATH),
        "n": summary["n"],
        "aggregate": {
            "dice":         round(summary["dice"],   4),
            "iou":          round(summary["iou"],    4),
            "cldice":       round(summary["cldice"], 4),
            "betti0_error": round(summary["betti0_error"], 2),
        },
    }, f, indent=2)
print(f"  JSON → {json_path}")

# ── zip overlays for download ─────────────────────────────────────────────────
zip_path = OUT_DIR / "overlays.zip"
print(f"\n  Zipping overlays ...", end=" ", flush=True)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for png in sorted(OVERLAY_DIR.glob("*.png")):
        zf.write(png, png.name)
print(f"done  → {zip_path}")
print(f"  ({zip_path.stat().st_size / 1e6:.1f} MB)")



  FIVES Test Set  —  200 images  —  epoch-26 checkpoint
  Dice        : 0.8022
  IoU         : 0.6824
  clDice      : 0.8290   ← headline
  Betti-0 err : 84.86
  n           : 200

  Per-category breakdown:
  Category                  n    Dice     IoU   clDice   B0 err
  --------------------------------------------------------------
  AMD                      50  0.8481  0.7397   0.8747    76.14
  DR                       50  0.8110  0.6885   0.8398    81.82
  Glaucoma                 50  0.7392  0.6141   0.7636    75.62
  Normal                   50  0.8107  0.6874   0.8378   105.88

  Worst case : test_122_G            clDice=0.1173
  Best  case : test_13_A             clDice=0.9480

  CSV  → /kaggle/working/fives_eval/per_image.csv
  JSON → /kaggle/working/fives_eval/summary.json

  Zipping overlays ... done  → /kaggle/working/fives_eval/overlays.zip
  (135.1 MB)


In [26]:
import os, sys, json, csv, zipfile

REPO = "/kaggle/working/LPG-SAM"
os.chdir(REPO)
sys.path.insert(0, REPO)

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from skimage.morphology import binary_closing, disk
from sklearn.metrics import roc_auc_score, average_precision_score

from src.adapter     import LatentPriorAdapter
from src.frangi      import fundus_to_frangi
from src.metrics     import MetricAccumulator, logits_to_binary
from src.sam_wrapper import FrozenSAM, SAM_INPUT_SIZE
from src.utils       import get_device, seed_everything

# ── paths ─────────────────────────────────────────────────────────────────────
# Adjust HRF_ROOT / STARE_ROOT if your dataset lives elsewhere in /kaggle/input
HRF_ROOT   = Path("/kaggle/working/HRF")      # images/ manual1/ mask/
STARE_ROOT = Path("/kaggle/working/stare")    # img/  gt/

MEDSAM_CKPT  = Path("checkpoints/medsam_vit_b.pth")
ADAPTER_CKPT = Path("checkpoints/frangi-updated-cldice-wt.pt")

# ── shared hyperparameters ────────────────────────────────────────────────────
with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

seed_everything(cfg["train"]["seed"])
device    = get_device()
THRESHOLD      = cfg["eval"]["binarize_threshold"]   # 0.5
CLOSING_RADIUS = cfg["eval"].get("closing_radius", 2)
PIXELS_PER_IMAGE = 8_192    # stratified subsample for AUC (RAM-safe)
rng = np.random.default_rng(42)

print(f"Device        : {device}")
print(f"Threshold     : {THRESHOLD}")
print(f"Closing radius: {CLOSING_RADIUS} px")

Device        : cuda
Threshold     : 0.5
Closing radius: 2 px


In [28]:
def load_rgb(path):
    """Open any image, convert to RGB, resize to SAM_INPUT_SIZE."""
    return np.array(
        Image.open(path).convert("RGB")
             .resize((SAM_INPUT_SIZE, SAM_INPUT_SIZE), Image.BICUBIC),
        dtype=np.uint8,
    )

def load_mask(path):
    """Open a binary mask, resize (NEAREST), return bool array."""
    return (
        np.array(
            Image.open(path).convert("L")
                 .resize((SAM_INPUT_SIZE, SAM_INPUT_SIZE), Image.NEAREST)
        ) > 127
    )

def normalise_for_sam(rgb_np):
    """(H,W,3) uint8 → (1,3,H,W) float32 tensor in [0,1]."""
    img = rgb_np.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0)

def compute_frangi(rgb_np):
    """(H,W,3) uint8 → (1,1,H,W) float32 tensor + (H,W) numpy map."""
    fng = fundus_to_frangi(rgb_np).astype(np.float32)
    return torch.from_numpy(fng).unsqueeze(0).unsqueeze(0), fng

def stratified_auc_sample(prob_np, gt_np, fov_np):
    """
    Sample PIXELS_PER_IMAGE/2 foreground + background pixels inside FOV.
    Returns (probs_1d, labels_1d) for sklearn AUC functions.
    """
    fg   = np.flatnonzero( gt_np & fov_np)
    bg   = np.flatnonzero(~gt_np & fov_np)
    half = PIXELS_PER_IMAGE // 2
    idx  = np.concatenate([
        rng.choice(fg, size=min(half, len(fg)), replace=False),
        rng.choice(bg, size=min(half, len(bg)), replace=False),
    ])
    flat = prob_np.ravel()
    gt_flat = gt_np.ravel()
    return flat[idx].astype(np.float32), gt_flat[idx].astype(np.uint8)

def find_file(directory, stem, exts=(".tif", ".jpg", ".JPG", ".jpeg", ".png", ".ppm")):
    for ext in exts:
        p = directory / f"{stem}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"No file for stem '{stem}' in {directory}")

def run_inference(rgb_np):
    """
    Full forward pass for one image.
    Returns (prob_np, pred_bin):
        prob_np  : (H,W) float32  — sigmoid probabilities
        pred_bin : (H,W) bool     — binarized + closed
    """
    img_t, (fng_t, _) = normalise_for_sam(rgb_np).to(device), compute_frangi(rgb_np)
    fng_t = fng_t.to(device)

    with torch.no_grad():
        z_img         = sam.encode_image(img_t)
        z_mod, tokens = adapter(z_img, fng_t)
        dense         = sam.zero_dense_prompt(1, device)
        low, _        = sam.decode_masks(z_mod, tokens, dense, multimask_output=False)
        full          = F.interpolate(low, size=(SAM_INPUT_SIZE, SAM_INPUT_SIZE),
                                      mode="bilinear", align_corners=False)

    prob_np  = torch.sigmoid(full).cpu().numpy()[0, 0]
    pred_bin = prob_np > THRESHOLD
    if CLOSING_RADIUS > 0:
        pred_bin = binary_closing(pred_bin, footprint=disk(CLOSING_RADIUS))
    return prob_np, pred_bin


def save_overlay(path, rgb_np, frangi_np, gt_np, pred_np, fov_np, stem, dice, cldice):
    """
    5-panel figure: Image | Frangi | GT | Prediction | TP/FP/FN overlay
    Metrics masked to FOV before display.
    """
    pred_v = pred_np & fov_np
    gt_v   = gt_np   & fov_np
    H, W   = gt_np.shape

    overlay = np.zeros((H, W, 3), np.float32)
    overlay[..., 1] = np.logical_and( pred_v,  gt_v).astype(np.float32)   # TP green
    overlay[..., 0] = np.logical_and( pred_v, ~gt_v).astype(np.float32)   # FP red
    overlay[..., 2] = np.logical_and(~pred_v,  gt_v).astype(np.float32)   # FN blue

    panels  = [rgb_np,    frangi_np,     gt_v,     pred_v,     overlay]
    titles  = ["Image",  "Frangi",  "Ground Truth", "Prediction", "TP=G  FP=R  FN=B"]
    kwargs  = [{},({"cmap":"hot","vmin":0,"vmax":1}),
               {"cmap":"gray","vmin":0,"vmax":1},
               {"cmap":"gray","vmin":0,"vmax":1},{}]

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    fig.suptitle(f"{stem}    Dice={dice:.3f}   clDice={cldice:.3f}", fontsize=11, y=1.01)
    for ax, im, title, kw in zip(axes, panels, titles, kwargs):
        ax.imshow(im, **kw)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.close(fig)


def zip_overlays(overlay_dir, zip_path):
    """Zip all PNGs in overlay_dir → zip_path."""
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(overlay_dir.glob("*.png")):
            zf.write(p, p.name)
    return zip_path.stat().st_size / 1e6   # MB


def print_summary(dataset_name, summary, auc_roc, auc_pr):
    print("\n" + "=" * 58)
    print(f"  Cross-dataset results — {dataset_name}")
    print("=" * 58)
    print(f"  n images    : {summary['n']}")
    print(f"  Dice        : {summary['dice']:.4f}")
    print(f"  IoU         : {summary['iou']:.4f}")
    print(f"  clDice      : {summary['cldice']:.4f}   ← headline")
    print(f"  Betti-0 err : {summary['betti0_error']:.2f}")
    print(f"  AUC-ROC     : {auc_roc:.4f}")
    print(f"  AUC-PR      : {auc_pr:.4f}")
    print("=" * 58)


def save_outputs(out_dir, dataset_name, acc, rows, auc_roc, auc_pr, category_rows=None):
    """Write summary.json and per_image.csv."""
    summary = acc.summary()

    # ── summary.json ──────────────────────────────────────────────────────────
    payload = {
        "dataset":          dataset_name,
        "checkpoint":       str(ADAPTER_CKPT),
        "checkpoint_epoch": ckpt.get("epoch"),
        "aggregate": {
            "dice":         round(summary["dice"],         4),
            "iou":          round(summary["iou"],          4),
            "cldice":       round(summary["cldice"],       4),
            "betti0_error": round(summary["betti0_error"], 2),
            "auc_roc":      round(auc_roc, 4),
            "auc_pr":       round(auc_pr,  4),
        },
    }
    if category_rows:
        payload["per_category"] = category_rows
    with open(out_dir / "summary.json", "w") as f:
        json.dump(payload, f, indent=2)

    # ── per_image.csv (worst-first by clDice) ─────────────────────────────────
    sorted_rows = sorted(rows, key=lambda r: r["cldice"])
    fieldnames  = list(sorted_rows[0].keys())
    with open(out_dir / "per_image.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(sorted_rows)

    return sorted_rows

In [29]:
HRF_ROOT   = Path("/kaggle/input/datasets/hirotakanifuji/hrf-dataset/HRF")      # images/ manual1/ mask/
STARE_ROOT = Path("/kaggle/input/datasets/hirotakanifuji/stare-dataset/STARE")    # img/  gt/

In [30]:
HRF_OUT = Path("/kaggle/working/hrf_eval")
(HRF_OUT / "overlays").mkdir(parents=True, exist_ok=True)

img_dir = HRF_ROOT / "images"
gt_dir  = HRF_ROOT / "manual1"
fov_dir = HRF_ROOT / "mask"

stems = sorted(
    p.stem for p in img_dir.iterdir()
    if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".tif")
)
print(f"HRF images found: {len(stems)}")

# category from stem suffix: 01_h→Healthy, 01_dr→DR, 01_g→Glaucoma
def hrf_category(stem):
    s = stem.lower()
    if s.endswith("_h"):  return "Healthy"
    if s.endswith("_dr"): return "Diabetic_Retinopathy"
    if s.endswith("_g"):  return "Glaucoma"
    return "Unknown"

acc_hrf    = MetricAccumulator()
rows_hrf   = []
all_probs_hrf, all_labels_hrf = [], []

print(f"\nRunning inference on {len(stems)} HRF images …")
for stem in tqdm(stems, ncols=90):
    rgb_np = load_rgb(find_file(img_dir, stem))

    # FOV mask (explicit .tif provided with HRF)
    fov_path = fov_dir / f"{stem}_mask.tif"
    fov_np   = load_mask(fov_path) if fov_path.exists() \
               else np.ones((SAM_INPUT_SIZE, SAM_INPUT_SIZE), bool)

    gt_np = load_mask(find_file(gt_dir, stem))

    # ── inference ──────────────────────────────────────────────────────────
    fng_t, fng_np = compute_frangi(rgb_np)
    img_t = normalise_for_sam(rgb_np).to(device)
    fng_t = fng_t.to(device)

    with torch.no_grad():
        z_img         = sam.encode_image(img_t)
        z_mod, tokens = adapter(z_img, fng_t)
        dense         = sam.zero_dense_prompt(1, device)
        low, _        = sam.decode_masks(z_mod, tokens, dense, multimask_output=False)
        full          = F.interpolate(low, size=(SAM_INPUT_SIZE, SAM_INPUT_SIZE),
                                      mode="bilinear", align_corners=False)

    prob_np  = torch.sigmoid(full).cpu().numpy()[0, 0]
    pred_bin = prob_np > THRESHOLD
    if CLOSING_RADIUS > 0:
        pred_bin = binary_closing(pred_bin, footprint=disk(CLOSING_RADIUS))

    # metrics (FOV-masked)
    acc_hrf.update(pred_bin & fov_np, gt_np & fov_np, name=stem)
    last = acc_hrf.per_image_table()[-1]
    rows_hrf.append({**last, "category": hrf_category(stem)})

    # AUC subsamples
    p_s, l_s = stratified_auc_sample(prob_np, gt_np, fov_np)
    all_probs_hrf.append(p_s); all_labels_hrf.append(l_s)

    # save overlay (ALL 45 images)
    save_overlay(
        path      = HRF_OUT / "overlays" / f"{stem}.png",
        rgb_np    = rgb_np,
        frangi_np = fng_np,
        gt_np     = gt_np,
        pred_np   = pred_bin,
        fov_np    = fov_np,
        stem      = stem,
        dice      = last["dice"],
        cldice    = last["cldice"],
    )
    torch.cuda.empty_cache()

# AUC
auc_roc_hrf = roc_auc_score(np.concatenate(all_labels_hrf),
                             np.concatenate(all_probs_hrf))
auc_pr_hrf  = average_precision_score(np.concatenate(all_labels_hrf),
                                      np.concatenate(all_probs_hrf))

print_summary("HRF (n=45)", acc_hrf.summary(), auc_roc_hrf, auc_pr_hrf)

# ── per-category breakdown ────────────────────────────────────────────────────
print("\n  Per-category:")
print(f"  {'Category':<24}  {'n':>3}  {'Dice':>6}  {'IoU':>6}  {'clDice':>7}  {'B0 err':>7}")
print("  " + "-" * 60)
hrf_cat_summary = {}
for cat in ["Healthy", "Diabetic_Retinopathy", "Glaucoma"]:
    cr = [r for r in rows_hrf if r["category"] == cat]
    if not cr: continue
    n   = len(cr)
    d   = np.mean([r["dice"]         for r in cr])
    iou = np.mean([r["iou"]          for r in cr])
    cl  = np.mean([r["cldice"]       for r in cr])
    b0  = np.mean([r["betti0_error"] for r in cr])
    print(f"  {cat:<24}  {n:>3}  {d:>6.4f}  {iou:>6.4f}  {cl:>7.4f}  {b0:>7.2f}")
    hrf_cat_summary[cat] = {"n":n,"dice":round(d,4),"iou":round(iou,4),
                             "cldice":round(cl,4),"betti0_error":round(b0,2)}

# ── worst / best ──────────────────────────────────────────────────────────────
sr = sorted(rows_hrf, key=lambda r: r["cldice"])
print(f"\n  Worst: {sr[0]['name']:<20} clDice={sr[0]['cldice']:.4f}  category={sr[0]['category']}")
print(f"  Best : {sr[-1]['name']:<20} clDice={sr[-1]['cldice']:.4f}  category={sr[-1]['category']}")

# ── save files + zip ──────────────────────────────────────────────────────────
save_outputs(HRF_OUT, "HRF", acc_hrf, rows_hrf, auc_roc_hrf, auc_pr_hrf, hrf_cat_summary)
mb = zip_overlays(HRF_OUT / "overlays", HRF_OUT / "overlays.zip")
print(f"\n  → {HRF_OUT}/summary.json")
print(f"  → {HRF_OUT}/per_image.csv")
print(f"  → {HRF_OUT}/overlays/  ({len(stems)} PNGs)")
print(f"  → {HRF_OUT}/overlays.zip  ({mb:.1f} MB)")

HRF images found: 45

Running inference on 45 HRF images …


  0%|                                                              | 0/45 [00:00<?, ?it/s]


  Cross-dataset results — HRF (n=45)
  n images    : 45
  Dice        : 0.7118
  IoU         : 0.5546
  clDice      : 0.7634   ← headline
  Betti-0 err : 293.69
  AUC-ROC     : 0.9583
  AUC-PR      : 0.9620

  Per-category:
  Category                    n    Dice     IoU   clDice   B0 err
  ------------------------------------------------------------
  Healthy                    15  0.7707  0.6272   0.8148    86.47
  Diabetic_Retinopathy       15  0.6789  0.5145   0.7300   380.67
  Glaucoma                   15  0.6858  0.5221   0.7454   413.93

  Worst: 06_dr                clDice=0.6403  category=Diabetic_Retinopathy
  Best : 15_h                 clDice=0.8771  category=Healthy

  → /kaggle/working/hrf_eval/summary.json
  → /kaggle/working/hrf_eval/per_image.csv
  → /kaggle/working/hrf_eval/overlays/  (45 PNGs)
  → /kaggle/working/hrf_eval/overlays.zip  (40.9 MB)


In [31]:
STARE_OUT = Path("/kaggle/working/stare_eval")
(STARE_OUT / "overlays").mkdir(parents=True, exist_ok=True)

img_dir_s = STARE_ROOT / "IMG"
gt_dir_s  = STARE_ROOT / "GT"

stems_s = sorted(
    p.stem for p in img_dir_s.iterdir()
    if p.suffix.lower() == ".ppm"
)
print(f"STARE images found: {len(stems_s)}")

def load_stare_gt(stem):
    """GT files are named {stem}.ah.ppm (double extension)."""
    return load_mask(gt_dir_s / f"{stem}.ah.ppm")

def auto_fov(rgb_np, threshold=10):
    """Derive FOV from bright pixels (STARE has no explicit masks)."""
    return np.any(rgb_np > threshold, axis=-1)

acc_stare    = MetricAccumulator()
rows_stare   = []
all_probs_s, all_labels_s = [], []

print(f"\nRunning inference on {len(stems_s)} STARE images …")
for stem in tqdm(stems_s, ncols=90):
    rgb_np = load_rgb(img_dir_s / f"{stem}.ppm")
    gt_np  = load_stare_gt(stem)
    fov_np = auto_fov(rgb_np)   # brightness-derived FOV

    fng_t, fng_np = compute_frangi(rgb_np)
    img_t = normalise_for_sam(rgb_np).to(device)
    fng_t = fng_t.to(device)

    with torch.no_grad():
        z_img         = sam.encode_image(img_t)
        z_mod, tokens = adapter(z_img, fng_t)
        dense         = sam.zero_dense_prompt(1, device)
        low, _        = sam.decode_masks(z_mod, tokens, dense, multimask_output=False)
        full          = F.interpolate(low, size=(SAM_INPUT_SIZE, SAM_INPUT_SIZE),
                                      mode="bilinear", align_corners=False)

    prob_np  = torch.sigmoid(full).cpu().numpy()[0, 0]
    pred_bin = prob_np > THRESHOLD
    if CLOSING_RADIUS > 0:
        pred_bin = binary_closing(pred_bin, footprint=disk(CLOSING_RADIUS))

    acc_stare.update(pred_bin & fov_np, gt_np & fov_np, name=stem)
    last = acc_stare.per_image_table()[-1]
    rows_stare.append({**last})

    p_s, l_s = stratified_auc_sample(prob_np, gt_np, fov_np)
    all_probs_s.append(p_s); all_labels_s.append(l_s)

    # save overlay (ALL 20 images)
    save_overlay(
        path      = STARE_OUT / "overlays" / f"{stem}.png",
        rgb_np    = rgb_np,
        frangi_np = fng_np,
        gt_np     = gt_np,
        pred_np   = pred_bin,
        fov_np    = fov_np,
        stem      = stem,
        dice      = last["dice"],
        cldice    = last["cldice"],
    )
    torch.cuda.empty_cache()

# AUC
auc_roc_s = roc_auc_score(np.concatenate(all_labels_s),
                           np.concatenate(all_probs_s))
auc_pr_s  = average_precision_score(np.concatenate(all_labels_s),
                                    np.concatenate(all_probs_s))

print_summary("STARE (n=20)", acc_stare.summary(), auc_roc_s, auc_pr_s)

# ── worst / best ──────────────────────────────────────────────────────────────
sr_s = sorted(rows_stare, key=lambda r: r["cldice"])
print(f"\n  Worst: {sr_s[0]['name']:<15} clDice={sr_s[0]['cldice']:.4f}")
print(f"  Best : {sr_s[-1]['name']:<15} clDice={sr_s[-1]['cldice']:.4f}")

# ── save files + zip ──────────────────────────────────────────────────────────
save_outputs(STARE_OUT, "STARE", acc_stare, rows_stare, auc_roc_s, auc_pr_s)
mb_s = zip_overlays(STARE_OUT / "overlays", STARE_OUT / "overlays_stare.zip")
print(f"\n  → {STARE_OUT}/summary.json")
print(f"  → {STARE_OUT}/per_image.csv")
print(f"  → {STARE_OUT}/overlays/  ({len(stems_s)} PNGs)")
print(f"  → {STARE_OUT}/overlays.zip  ({mb_s:.1f} MB)")


# ── CELL 6 ── Side-by-side comparison table ───────────────────────────────────
hrf_s   = acc_hrf.summary()
stare_s = acc_stare.summary()

print("\n" + "=" * 72)
print("  Full cross-dataset comparison")
print("=" * 72)
print(f"  {'Metric':<14}  {'FIVES (in-dist)':>16}  {'HRF (zero-shot)':>16}  {'STARE (zero-shot)':>18}")
print("  " + "-" * 68)
fives = {"dice":0.8023,"iou":0.6824,"cldice":0.8290,"betti0_error":84.88}
rows_cmp = [
    ("Dice",        fives["dice"],         hrf_s["dice"],         stare_s["dice"]),
    ("IoU",         fives["iou"],          hrf_s["iou"],          stare_s["iou"]),
    ("clDice",      fives["cldice"],       hrf_s["cldice"],       stare_s["cldice"]),
    ("Betti-0 err", fives["betti0_error"], hrf_s["betti0_error"], stare_s["betti0_error"]),
    ("AUC-ROC",     "—",                   f"{auc_roc_hrf:.4f}",  f"{auc_roc_s:.4f}"),
    ("AUC-PR",      "—",                   f"{auc_pr_hrf:.4f}",   f"{auc_pr_s:.4f}"),
]
for label, f, h, s in rows_cmp:
    fv = f"{f:.4f}" if isinstance(f, float) else f
    hv = f"{h:.4f}" if isinstance(h, float) else h
    sv = f"{s:.4f}" if isinstance(s, float) else s
    print(f"  {label:<14}  {fv:>16}  {hv:>16}  {sv:>18}")
print("=" * 72)


STARE images found: 20

Running inference on 20 STARE images …


  0%|                                                              | 0/20 [00:00<?, ?it/s]


  Cross-dataset results — STARE (n=20)
  n images    : 20
  Dice        : 0.6831
  IoU         : 0.5202
  clDice      : 0.7488   ← headline
  Betti-0 err : 76.95
  AUC-ROC     : 0.9617
  AUC-PR      : 0.9571

  Worst: im0324          clDice=0.6654
  Best : im0255          clDice=0.8036

  → /kaggle/working/stare_eval/summary.json
  → /kaggle/working/stare_eval/per_image.csv
  → /kaggle/working/stare_eval/overlays/  (20 PNGs)
  → /kaggle/working/stare_eval/overlays.zip  (17.4 MB)

  Full cross-dataset comparison
  Metric           FIVES (in-dist)   HRF (zero-shot)   STARE (zero-shot)
  --------------------------------------------------------------------
  Dice                      0.8023            0.7118              0.6831
  IoU                       0.6824            0.5546              0.5202
  clDice                    0.8290            0.7634              0.7488
  Betti-0 err              84.8800          293.6889             76.9500
  AUC-ROC                        —            

In [32]:
import numpy as np
import yaml
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm
from skimage.morphology import binary_closing, disk

from src.dataset import FundusVesselDataset
from src.metrics import MetricAccumulator
from src.utils import seed_everything

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)
seed_everything(cfg["train"]["seed"])

CLOSING_RADIUS  = cfg["eval"].get("closing_radius", 0)
THRESHOLDS      = np.arange(0.05, 0.96, 0.05).tolist()   # 19 candidates

def make_loader(split_file, batch_size=4):
    ds = FundusVesselDataset(
        data_root  = "data/processed/FIVES",
        split_file = split_file,
        augment    = False,
        seed       = cfg["train"]["seed"],
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=2, pin_memory=False)

val_loader  = make_loader("data/processed/FIVES/val.txt")
test_loader = make_loader("data/processed/FIVES/test.txt")

def score_frangi(loader, threshold, desc=""):
    acc = MetricAccumulator()
    for batch in loader:
        frangi = batch["frangi"].numpy()   # (B,1,H,W) float32 in [0,1]
        gt     = batch["mask"].numpy()     # (B,1,H,W) float32
        names  = batch["name"]
        for b in range(frangi.shape[0]):
            pred = frangi[b, 0] > threshold
            if CLOSING_RADIUS > 0:
                pred = binary_closing(pred, footprint=disk(CLOSING_RADIUS))
            gt_b = gt[b, 0] > 0.5
            acc.update(pred, gt_b, name=names[b])
    return acc

print("Sweeping thresholds on val set...")
print(f"{'Threshold':>10}  {'Dice':>7}  {'IoU':>7}  {'clDice':>8}  {'B0err':>8}")
print("─" * 50)

best_cldice, best_thresh = -1, None
for t in tqdm(THRESHOLDS, desc="threshold sweep", ncols=70):
    acc = score_frangi(val_loader, threshold=t)
    s   = acc.summary()
    marker = "  ←" if s["cldice"] > best_cldice else ""
    print(f"  t={t:.2f}      {s['dice']:.4f}  {s['iou']:.4f}  "
          f"{s['cldice']:.4f}  {s['betti0_error']:8.2f}{marker}")
    if s["cldice"] > best_cldice:
        best_cldice  = s["cldice"]
        best_thresh  = t

print(f"\n→ Best val threshold: {best_thresh:.2f}  (val clDice={best_cldice:.4f})")

print(f"\nEvaluating on test set at threshold={best_thresh:.2f} ...")
test_acc = score_frangi(test_loader, threshold=best_thresh, desc="test")
s = test_acc.summary()

print("\n" + "=" * 50)
print(f"  Frangi threshold baseline  (t={best_thresh:.2f})")
print("=" * 50)
print(f"  n samples  : {s['n']}")
print(f"  Dice       : {s['dice']:.4f}")
print(f"  IoU        : {s['iou']:.4f}")
print(f"  clDice     : {s['cldice']:.4f}    <- headline metric")
print(f"  Betti-0 err: {s['betti0_error']:.2f}")
print("=" * 50)


Sweeping thresholds on val set...
 Threshold     Dice      IoU    clDice     B0err
──────────────────────────────────────────────────


threshold sweep:   5%|▉                | 1/19 [00:17<05:23, 17.97s/it]

  t=0.05      0.1969  0.1097  0.2164     21.97  ←


threshold sweep:  11%|█▊               | 2/19 [00:31<04:18, 15.20s/it]

  t=0.10      0.2533  0.1458  0.2286   1129.88  ←


...

threshold sweep:  95%|███████████████▏| 18/19 [03:05<00:09,  9.28s/it]

  t=0.90      0.0002  0.0001  0.0001    147.58


threshold sweep: 100%|████████████████| 19/19 [03:14<00:00, 10.21s/it]

  t=0.95      0.0001  0.0000  0.0000    117.48

→ Best val threshold: 0.15  (val clDice=0.2828)

Evaluating on test set at threshold=0.15 ...



  Frangi threshold baseline  (t=0.15)
  n samples  : 200
  Dice       : 0.3129
  IoU        : 0.1932
  clDice     : 0.2840    <- headline metric
  Betti-0 err: 4872.30


**UNET EVAL STARTS HERE**

In [34]:
import os
import shutil

# Source: Kaggle dataset path
src_path = '/kaggle/input/datasets/hirotakanifuji/unet-checkpoint/ckpt_best.pt'  # adjust dataset name if needed

# Destination: your repo checkpoints folder
dst_dir = '/kaggle/working/LPG-SAM/checkpoints/'
dst_path = os.path.join(dst_dir, 'ckpt_best.pt')

# Ensure destination directory exists
os.makedirs(dst_dir, exist_ok=True)

# Copy file
shutil.copy(src_path, dst_path)

print(f"Checkpoint copied to: {dst_path}")

Checkpoint copied to: /kaggle/working/LPG-SAM/checkpoints/ckpt_best.pt


In [35]:
import os
import shutil

# Source: Kaggle dataset path
src_path = '/kaggle/input/datasets/hirotakanifuji/unet-code/train_fives.py'  # adjust dataset name if needed

# Destination: your repo checkpoints folder
dst_dir = '/kaggle/working/LPG-SAM/'
dst_path = os.path.join(dst_dir, 'train_fives.py')

# Ensure destination directory exists
os.makedirs(dst_dir, exist_ok=True)

# Copy file
shutil.copy(src_path, dst_path)

print(f"Checkpoint copied to: {dst_path}")

Checkpoint copied to: /kaggle/working/LPG-SAM/train_fives.py


In [38]:
import os, sys, json, csv, zipfile

# train_fives.py lives directly in /kaggle/working/
sys.path.insert(0, "/kaggle/working")
sys.path.insert(0, "/kaggle/working/LPG-SAM")   # src.metrics

# Patch tensorboard so the import inside train_fives.py doesn't crash if missing
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    import types
    tb_mod = types.ModuleType("torch.utils.tensorboard")
    tb_mod.SummaryWriter = type("SummaryWriter", (), {"__init__": lambda *a, **k: None,
                                                       "add_scalar": lambda *a, **k: None,
                                                       "close": lambda *a, **k: None})
    sys.modules["torch.utils.tensorboard"] = tb_mod

from train_fives import DualBranchVesselNet   # model class
from src.metrics import MetricAccumulator     # Dice / IoU / clDice / Betti-0

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from skimage.morphology import binary_closing, disk

# ── paths ─────────────────────────────────────────────────────────────────────
CKPT_PATH   = Path("/kaggle/working/LPG-SAM/checkpoints/ckpt_best.pt")
MEDSAM_CKPT = Path("/kaggle/working/LPG-SAM/checkpoints/medsam_vit_b.pth")
DATA_ROOT   = Path("/kaggle/working/LPG-SAM/data/processed/FIVES")
SPLIT_FILE  = DATA_ROOT / "test.txt"
OUT_DIR     = Path("/kaggle/working/dualbranch_eval")
OVERLAY_DIR = OUT_DIR / "overlays"
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

# ── hyperparameters ────────────────────────────────────────────────────────────
THRESHOLD      = 0.5
CLOSING_RADIUS = 2        # morphological closing radius (px, at model resolution)
CAT_MAP = {"A": "AMD", "D": "Diabetic_Retinopathy", "G": "Glaucoma", "N": "Normal"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stems  = [l.strip() for l in SPLIT_FILE.read_text().splitlines() if l.strip()]
print(f"Device      : {device}")
print(f"Checkpoint  : {CKPT_PATH}")
print(f"Test images : {len(stems)}")

Device      : cuda
Checkpoint  : /kaggle/working/LPG-SAM/checkpoints/ckpt_best.pt
Test images : 200


In [39]:
state = torch.load(CKPT_PATH, map_location=device)
train_cfg = state.get("config", {})

# Read training config so we use exactly the same img_size / sigma
IMG_SIZE           = train_cfg.get("img_size",            512)
SIGMA              = train_cfg.get("sigma",               2.0)
FREEZE_UNTIL       = train_cfg.get("freeze_encoder_until", 10)

print(f"img_size    : {IMG_SIZE}")
print(f"sigma       : {SIGMA}")
print(f"Building DualBranchVesselNet ...")

model = DualBranchVesselNet(
    medsam_ckpt          = MEDSAM_CKPT,
    img_size             = IMG_SIZE,
    freeze_encoder_until = FREEZE_UNTIL,
    sigma                = SIGMA,
).to(device)

model.load_state_dict(state["model"])
model.eval()

n_total   = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Checkpoint epoch   : {state.get('epoch', '?')}")
print(f"Checkpoint val dice: {state.get('val_dice', float('nan')):.4f}")
print(f"Total params       : {n_total/1e6:.2f}M   trainable: {n_trainable/1e6:.2f}M")


img_size    : 512
sigma       : 2.0
Building DualBranchVesselNet ...
[MedSAM] Resizing pos_embed (1, 64, 64, 768) -> (1, 32, 32, 768)
[MedSAM] Missing keys (first 5): ['blocks.0.attn.rel_pos_h', 'blocks.0.attn.rel_pos_w', 'blocks.1.attn.rel_pos_h', 'blocks.1.attn.rel_pos_w', 'blocks.2.attn.rel_pos_h'] (total 24)
Checkpoint epoch   : 47
Checkpoint val dice: 0.9353
Total params       : 93.28M   trainable: 20.98M


In [40]:
# ImageNet normalization (must match FIVESDataset in train_fives.py)
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def load_image_1024(path):
    """Load PNG → (1024,1024) uint8 RGB numpy."""
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)

def load_mask_1024(path):
    """Load PNG → (1024,1024) bool numpy."""
    return np.array(Image.open(path).convert("L"), dtype=np.uint8) > 127

def preprocess(rgb_np):
    """
    Resize to IMG_SIZE, apply ImageNet norm.
    Returns (1, 3, IMG_SIZE, IMG_SIZE) float32 tensor.
    """
    pil = Image.fromarray(rgb_np).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    t   = torch.from_numpy(np.array(pil)).permute(2, 0, 1).float() / 255.0
    t   = (t - _MEAN) / _STD
    return t.unsqueeze(0)   # (1, 3, H, H)

def to_display(rgb_np, size):
    """Resize uint8 RGB for display at `size`."""
    return np.array(Image.fromarray(rgb_np).resize((size, size), Image.BILINEAR))

def sigmoid_np(logit_t):
    """(1,1,H,W) tensor → (H,W) float32 numpy probabilities."""
    return torch.sigmoid(logit_t).cpu().numpy()[0, 0]

def binarize(prob_np, radius=CLOSING_RADIUS):
    p = prob_np > THRESHOLD
    if radius > 0:
        p = binary_closing(p, footprint=disk(radius))
    return p

def save_overlay(path, rgb_np, gt_np, pred_np,
                 prob_low, prob_high, alpha_np,
                 dice_v, cldice_v, stem):
    """
    7-panel overlay (all panels shown at native model resolution IMG_SIZE):
      Image | GT | Fused pred | Global branch | Local branch | Alpha gate | TP/FP/FN
    """
    sz = IMG_SIZE
    img_disp  = to_display(rgb_np, sz)       # uint8 RGB
    gt_disp   = gt_np                        # bool (IMG_SIZE, IMG_SIZE)
    pred_disp = pred_np

    # TP/FP/FN color overlay
    H, W = gt_disp.shape
    ov = np.zeros((H, W, 3), np.float32)
    ov[..., 1] = np.logical_and( pred_disp,  gt_disp).astype(np.float32)  # TP green
    ov[..., 0] = np.logical_and( pred_disp, ~gt_disp).astype(np.float32)  # FP red
    ov[..., 2] = np.logical_and(~pred_disp,  gt_disp).astype(np.float32)  # FN blue

    panels = [img_disp, gt_disp, pred_disp, prob_low, prob_high, alpha_np, ov]
    titles = [
        "Image",
        "Ground Truth",
        f"Fused pred\nDice={dice_v:.3f}  clDice={cldice_v:.3f}",
        "Global branch\n(low-freq ViT)",
        "Local branch\n(high-freq UNet)",
        "Alpha gate\n(high=UNet)",
        "TP=G  FP=R  FN=B",
    ]
    cmaps = [None, "gray", "gray", "hot", "hot", "RdYlGn", None]
    vmins = [None, 0,    0,    0, 0, 0, None]
    vmaxs = [None, 1,    1,    1, 1, 1, None]

    fig, axes = plt.subplots(1, 7, figsize=(35, 5))
    fig.suptitle(stem, fontsize=11, y=1.01)
    for ax, im, title, cmap, vmin, vmax in zip(axes, panels, titles, cmaps, vmins, vmaxs):
        ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    fig.savefig(path, dpi=90, bbox_inches="tight")
    plt.close(fig)


In [41]:
acc  = MetricAccumulator()
rows = []

print(f"Running inference on {len(stems)} FIVES test images at {IMG_SIZE}px …\n")

with torch.no_grad():
    for stem in tqdm(stems, ncols=90):
        # ── load ──────────────────────────────────────────────────────────────
        rgb_np   = load_image_1024(DATA_ROOT / "images_1024" / f"{stem}.png")
        mask_1024 = load_mask_1024(DATA_ROOT / "masks_1024"  / f"{stem}.png")

        # Resize GT to model resolution for metric computation
        gt_np = np.array(
            Image.fromarray(mask_1024.astype(np.uint8) * 255)
                 .resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
        ) > 127

        # ── forward pass ──────────────────────────────────────────────────────
        img_t = preprocess(rgb_np).to(device)
        out   = model(img_t)

        prob_fused = sigmoid_np(out["logit"])       # (IMG_SIZE, IMG_SIZE)
        prob_low   = sigmoid_np(out["logit_low"])
        prob_high  = sigmoid_np(out["logit_high"])
        alpha_np   = out["alpha"].cpu().numpy()[0, 0]

        pred_bin = binarize(prob_fused)

        # ── metrics ───────────────────────────────────────────────────────────
        acc.update(pred_bin, gt_np, name=stem)
        last = acc.per_image_table()[-1]
        rows.append({**last, "category": CAT_MAP.get(stem[-1], "Unknown")})

        # ── save overlay ──────────────────────────────────────────────────────
        save_overlay(
            path      = OVERLAY_DIR / f"{stem}.png",
            rgb_np    = rgb_np,
            gt_np     = gt_np,
            pred_np   = pred_bin,
            prob_low  = prob_low,
            prob_high = prob_high,
            alpha_np  = alpha_np,
            dice_v    = last["dice"],
            cldice_v  = last["cldice"],
            stem      = stem,
        )
        torch.cuda.empty_cache()

print(f"\nOverlays saved: {len(list(OVERLAY_DIR.glob('*.png')))}")

Running inference on 200 FIVES test images at 512px …



  0%|                                                             | 0/200 [00:00<?, ?it/s]


Overlays saved: 200


In [42]:
summary = acc.summary()

# ── aggregate ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print(f"  DualBranchVesselNet  —  FIVES test set  —  200 images  ({IMG_SIZE}px)")
print("=" * 62)
print(f"  Dice        : {summary['dice']:.4f}")
print(f"  IoU         : {summary['iou']:.4f}")
print(f"  clDice      : {summary['cldice']:.4f}   ← headline")
print(f"  Betti-0 err : {summary['betti0_error']:.2f}")
print(f"  n           : {summary['n']}")

# ── per-category ──────────────────────────────────────────────────────────────
print(f"\n  Per-category breakdown:")
print(f"  {'Category':<24} {'n':>4}  {'Dice':>6}  {'IoU':>6}  {'clDice':>7}  {'B0 err':>7}")
print("  " + "-" * 62)
cat_summary = {}
for cat_key, cat_label in [("AMD","AMD"), ("Diabetic_Retinopathy","DR"),
                            ("Glaucoma","Glaucoma"), ("Normal","Normal")]:
    cr = [r for r in rows if r["category"] == cat_key]
    if not cr: continue
    n   = len(cr)
    d   = np.mean([r["dice"]         for r in cr])
    iou = np.mean([r["iou"]          for r in cr])
    cl  = np.mean([r["cldice"]       for r in cr])
    b0  = np.mean([r["betti0_error"] for r in cr])
    print(f"  {cat_label:<24} {n:>4}  {d:>6.4f}  {iou:>6.4f}  {cl:>7.4f}  {b0:>7.2f}")
    cat_summary[cat_key] = {"n":n, "dice":round(d,4), "iou":round(iou,4),
                             "cldice":round(cl,4), "betti0_error":round(b0,2)}

# ── worst / best ──────────────────────────────────────────────────────────────
sr = sorted(rows, key=lambda r: r["cldice"])
print(f"\n  Worst : {sr[0]['name']:<20}  clDice={sr[0]['cldice']:.4f}  ({sr[0]['category']})")
print(f"  Best  : {sr[-1]['name']:<20}  clDice={sr[-1]['cldice']:.4f}  ({sr[-1]['category']})")

# ── comparison vs LPG-SAM (fill in your numbers below) ───────────────────────
lpg = {"dice": 0.8023, "iou": 0.6824, "cldice": 0.8290, "betti0_error": 84.88}
print(f"\n  {'Metric':<14}  {'LPG-SAM':>10}  {'DualBranch':>11}  {'Δ':>8}")
print("  " + "-" * 48)
for k, label in [("dice","Dice"), ("iou","IoU"), ("cldice","clDice"), ("betti0_error","Betti-0 err")]:
    delta = summary[k] - lpg[k]
    sign  = "+" if delta >= 0 else ""
    print(f"  {label:<14}  {lpg[k]:>10.4f}  {summary[k]:>11.4f}  {sign}{delta:>7.4f}")

# ── save per-image CSV ────────────────────────────────────────────────────────
csv_path = OUT_DIR / "per_image.csv"
sorted_rows = sorted(rows, key=lambda r: r["cldice"])   # worst-first
with open(csv_path, "w", newline="") as f:
    fieldnames = ["name", "category", "dice", "iou", "cldice", "betti0_error"]
    w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    w.writeheader(); w.writerows(sorted_rows)

# ── save summary JSON ─────────────────────────────────────────────────────────
json_path = OUT_DIR / "summary.json"
with open(json_path, "w") as f:
    json.dump({
        "model":            "DualBranchVesselNet",
        "checkpoint":       str(CKPT_PATH),
        "checkpoint_epoch": state.get("epoch"),
        "img_size":         IMG_SIZE,
        "aggregate": {
            "dice":         round(summary["dice"],   4),
            "iou":          round(summary["iou"],    4),
            "cldice":       round(summary["cldice"], 4),
            "betti0_error": round(summary["betti0_error"], 2),
        },
        "per_category": cat_summary,
    }, f, indent=2)

# ── zip overlays ──────────────────────────────────────────────────────────────
zip_path = OUT_DIR / "overlays.zip"
print(f"\n  Zipping 200 overlays …", end=" ", flush=True)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OVERLAY_DIR.glob("*.png")):
        zf.write(p, p.name)
mb = zip_path.stat().st_size / 1e6
print(f"done  ({mb:.1f} MB)")

print(f"\n  → {csv_path}")
print(f"  → {json_path}")
print(f"  → {OVERLAY_DIR}/  (200 PNGs)")
print(f"  → {zip_path}")



  DualBranchVesselNet  —  FIVES test set  —  200 images  (512px)
  Dice        : 0.8550
  IoU         : 0.7538
  clDice      : 0.8840   ← headline
  Betti-0 err : 21.67
  n           : 200

  Per-category breakdown:
  Category                    n    Dice     IoU   clDice   B0 err
  --------------------------------------------------------------
  AMD                        50  0.8769  0.7820   0.9069    20.40
  DR                         50  0.8632  0.7619   0.8896    17.58
  Glaucoma                   50  0.8212  0.7170   0.8506    13.74
  Normal                     50  0.8588  0.7541   0.8890    34.96

  Worst : test_123_G            clDice=0.1125  (Glaucoma)
  Best  : test_70_D             clDice=0.9479  (Diabetic_Retinopathy)

  Metric             LPG-SAM   DualBranch         Δ
  ------------------------------------------------
  Dice                0.8023       0.8550  + 0.0527
  IoU                 0.6824       0.7538  + 0.0714
  clDice              0.8290       0.8840  + 0.0550